# Predicting Student Health Risk - Baseline Modeling

This notebook is the public baseline and compact model-comparison notebook for S6E7. Instead of publishing many small notebooks, we keep the baseline story in one place: data loading, preprocessing, validation, weighting comparison, champion selection, output diagnostics, and `submission.csv` generation.

The first submitted baseline used fully balanced class weights and scored `0.90603` on the public leaderboard. Its main lesson was clear: minority recall was strong, but `fit` and `unhealthy` were over-predicted. This version compares less aggressive weighting strategies before choosing the next baseline champion.

## 1. Setup

The notebook is designed to run both on Kaggle and locally. Kaggle competition inputs can mount under different folder names, so the resolver scans `/kaggle/input` if the common paths are absent.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

SEED = 42
N_SPLITS = 5

candidate_dirs = [
    Path('/kaggle/input/playground-series-s6e7'),
    Path('/kaggle/input/predicting-student-health-risk'),
    Path('../data'),
    Path('data'),
]
DATA_DIR = next(
    (path for path in candidate_dirs if (path / 'train.csv').exists()),
    None,
)
if DATA_DIR is None and Path('/kaggle/input').exists():
    for train_path in Path('/kaggle/input').glob('**/train.csv'):
        parent = train_path.parent
        if (parent / 'test.csv').exists() and (parent / 'sample_submission.csv').exists():
            DATA_DIR = parent
            break
if DATA_DIR is None:
    raise FileNotFoundError('Could not locate train/test/sample_submission CSV files.')

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('..')
PRED_DIR = OUTPUT_DIR / 'predictions'
PRED_DIR.mkdir(exist_ok=True)
print(f'Using data directory: {DATA_DIR}')

## 2. Load Data And Define The Target

The target is the only train column that does not appear in test: `health_condition`. The sample submission defines the expected ID and output column order.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

id_col = sample_submission.columns[0]
target_candidates = [col for col in train.columns if col not in test.columns]
target = target_candidates[0]

feature_cols = [col for col in test.columns if col in train.columns and col != id_col]
X = train[feature_cols]
y = train[target]
X_test = test[feature_cols]

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [col for col in feature_cols if col not in num_cols]
classes = np.array(sorted(y.unique()))
target_share = y.value_counts(normalize=True).sort_index()

print(f'train shape: {train.shape}')
print(f'test shape: {test.shape}')
print(f'target: {target}')
print(f'numeric features ({len(num_cols)}): {num_cols}')
print(f'categorical features ({len(cat_cols)}): {cat_cols}')
display(y.value_counts().to_frame('count'))
display(target_share.to_frame('target_share'))

## 3. Validation Design

The leaderboard scores are around an accuracy-like scale, while the target is very imbalanced. We therefore compare candidates by accuracy first, but keep balanced accuracy and macro F1 as guardrails so the minority classes do not disappear.

The key baseline question is no longer “does class balancing help?” It is “how much class balancing is too much?”

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
print(f'Using {N_SPLITS}-fold StratifiedKFold with seed={SEED}.')

## 4. Shared Preprocessing

All candidates use the same fold-safe preprocessing:

- numeric median imputation plus missing-value indicators;
- categorical most-frequent imputation plus ordinal encoding;
- identical HGB settings except for fold-level sample weighting.

This isolates the comparison to the class-prior strategy.

In [ ]:
def make_preprocess() -> ColumnTransformer:
    """Create the shared fold-safe preprocessing pipeline."""
    return ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median', add_indicator=True), num_cols),
            (
                'cat',
                Pipeline(
                    [
                        ('imputer', SimpleImputer(strategy='most_frequent')),
                        (
                            'encoder',
                            OrdinalEncoder(
                                handle_unknown='use_encoded_value',
                                unknown_value=-1,
                            ),
                        ),
                    ]
                ),
                cat_cols,
            ),
        ],
        remainder='drop',
    )


def make_pipeline() -> Pipeline:
    """Create a candidate HGB pipeline. Weighting is passed at fit time."""
    model = HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=160,
        l2_regularization=0.05,
        early_stopping=True,
        random_state=SEED,
    )
    return Pipeline(
        [
            ('preprocess', make_preprocess()),
            ('model', model),
        ]
    )

candidate_configs = [
    {
        'name': 'hgb_unweighted',
        'weighting': 'none',
        'weight_map': None,
        'description': 'Accuracy-oriented reference without class weighting.',
    },
    {
        'name': 'hgb_light_weight',
        'weighting': 'custom',
        'weight_map': {'at-risk': 1.0, 'fit': 1.8, 'unhealthy': 1.6},
        'description': 'Moderate minority support without fully balancing priors.',
    },
    {
        'name': 'hgb_balanced',
        'weighting': 'balanced',
        'weight_map': None,
        'description': 'Previous public baseline with full class balancing.',
    },
]

display(pd.DataFrame(candidate_configs)[['name', 'description']])

## 5. Cross-Validated Candidate Comparison

Each candidate is trained across the same folds. We store OOF predictions, fold metrics, test predictions, and prediction shares for every candidate. The comparison table is the main decision point before touching the leaderboard again.

In [ ]:
def evaluate_candidate(config: dict) -> dict:
    """Run CV for one candidate and return metrics plus predictions."""
    name = config['name']
    pipeline_template = make_pipeline()
    oof_pred = pd.Series(index=train.index, dtype=object)
    test_votes = []
    fold_rows = []

    for fold, (trn_idx, val_idx) in enumerate(cv.split(X, y), start=1):
        pipeline = clone(pipeline_template)
        X_trn, X_val = X.iloc[trn_idx], X.iloc[val_idx]
        y_trn, y_val = y.iloc[trn_idx], y.iloc[val_idx]

        if config['weighting'] == 'balanced':
            sample_weight = compute_sample_weight('balanced', y_trn)
        elif config['weighting'] == 'custom':
            sample_weight = y_trn.map(config['weight_map']).to_numpy()
        else:
            sample_weight = None

        fit_params = {}
        if sample_weight is not None:
            fit_params['model__sample_weight'] = sample_weight
        pipeline.fit(X_trn, y_trn, **fit_params)
        val_pred = pipeline.predict(X_val)
        test_votes.append(pipeline.predict(X_test))
        oof_pred.iloc[val_idx] = val_pred

        fold_rows.append(
            {
                'candidate': name,
                'fold': fold,
                'accuracy': accuracy_score(y_val, val_pred),
                'balanced_accuracy': balanced_accuracy_score(y_val, val_pred),
                'macro_f1': f1_score(y_val, val_pred, average='macro'),
                'weighted_f1': f1_score(y_val, val_pred, average='weighted'),
            }
        )

    vote_frame = pd.DataFrame(np.column_stack(test_votes))
    test_pred = vote_frame.mode(axis=1)[0]
    fold_metrics = pd.DataFrame(fold_rows)
    oof_share = oof_pred.value_counts(normalize=True).reindex(classes).fillna(0.0)
    test_share = test_pred.value_counts(normalize=True).reindex(classes).fillna(0.0)

    print(f'Completed {name}')
    display(fold_metrics.groupby('candidate').mean(numeric_only=True))

    return {
        'name': name,
        'fold_metrics': fold_metrics,
        'oof_pred': oof_pred,
        'test_pred': test_pred,
        'oof_share': oof_share,
        'test_share': test_share,
    }

candidate_results = [evaluate_candidate(config) for config in candidate_configs]

## 6. Select The Baseline Champion

A strong candidate should improve accuracy without producing an obviously broken class mix. Because the first leaderboard result showed the fully balanced model over-predicts minority classes, prediction share is part of the decision, not just a postscript.

In [ ]:
metric_table = pd.concat([result['fold_metrics'] for result in candidate_results])
summary = metric_table.groupby('candidate').agg(
    accuracy=('accuracy', 'mean'),
    balanced_accuracy=('balanced_accuracy', 'mean'),
    macro_f1=('macro_f1', 'mean'),
    weighted_f1=('weighted_f1', 'mean'),
).sort_values('accuracy', ascending=False)

share_rows = []
for result in candidate_results:
    row = {'candidate': result['name']}
    for cls in classes:
        row[f'oof_{cls}_share'] = result['oof_share'].loc[cls]
        row[f'test_{cls}_share'] = result['test_share'].loc[cls]
    share_rows.append(row)
share_table = pd.DataFrame(share_rows).set_index('candidate')

comparison = summary.join(share_table)
display(comparison)

champion_name = comparison.index[0]
champion = next(result for result in candidate_results if result['name'] == champion_name)
print(f'Selected champion by mean CV accuracy: {champion_name}')

## 7. Champion Diagnostics

The champion gets the full classification report and confusion matrix. If it wins by accuracy but collapses minority recall, it should be treated as a leaderboard probe rather than a final modeling direction.

In [ ]:
champion_oof = champion['oof_pred']
print(classification_report(y, champion_oof, digits=4))

cm = pd.DataFrame(
    confusion_matrix(y, champion_oof, labels=classes),
    index=[f'true_{cls}' for cls in classes],
    columns=[f'pred_{cls}' for cls in classes],
)
display(cm)

pd.DataFrame(
    {
        id_col: train[id_col] if id_col in train else train.index,
        'oof_pred': champion_oof,
        'target': y,
    }
).to_csv(PRED_DIR / f'{champion_name}_oof_predictions.csv', index=False)

comparison.to_csv(OUTPUT_DIR / 'baseline_model_comparison.csv')

## 8. Create The Submission

The notebook writes one `submission.csv`, generated by the selected champion. This keeps the public notebook as the source of truth for leaderboard submissions.

In [ ]:
submission = sample_submission.copy()
submission.iloc[:, 1] = champion['test_pred'].values
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)

display(submission.head())
display(submission.iloc[:, 1].value_counts(normalize=True).to_frame('prediction_share'))
print(f'Wrote submission to {OUTPUT_DIR / "submission.csv"}')
print(f'Champion candidate: {champion_name}')

## 9. Public Baseline History

The earlier public baseline used `hgb_balanced` and scored `0.90603`. It achieved strong minority recall but predicted `fit` and `unhealthy` almost twice as often as their training prevalence.

This comparison keeps that result as an anchor while testing whether a less aggressive prior improves the accuracy-like leaderboard direction.

## 10. Next Modeling Moves

- If `hgb_unweighted` or `hgb_light_weight` beats `hgb_balanced`, submit the notebook-generated `submission.csv` as the next baseline probe.
- Add CatBoost with native categorical handling in this same notebook or as a clearly separated optional section.
- Add LightGBM only after the weighting question is answered.
- Keep using notebook-generated submissions rather than detached local files.